In [ ]:
%sql
/* ===================================================================================
   VIEW: vw_cost_center_mapping_bootstrap
   SOURCE: hive_metastore.ra_adido_2025.fy25_cost_center_mapping
   
   BUSINESS RULES & LOGIC APPLIED:
   
   1. MUTUALLY EXCLUSIVE BRANCHING (The "Yes" Rule) - Case Insensitive:
      - If AdditionalAssessableUnitIDandNameandSegment is 'YES' (or blank): Use ONLY 
        Primary Columns (AssessableUnitID, AssessableUnitName, Segment). 
        Ignore the Additional columns.
      - If AdditionalAssessableUnitIDandNameandSegment is NOT 'YES': Use ONLY 
        Additional columns. Ignore Primary entirely.
      
   2. DIRTY STRING PARSING (Mashed Concatenations):
      - Additional columns often contain multiple Assessable Units mashed together 
        without spacing (e.g., "123456 - Name - Seg123457 - Name - Seg").
      - Uses Regex to slice the string into individual blocks strictly at every 
        6-digit boundary.
      
   3. HYPHEN PROTECTION & FALLBACK:
      - Extracts the 6-digit AU_ID.
      - For the remaining text: If hyphens exist, splits Name and Segment based on the 
        LAST hyphen (protecting internal hyphens like "Retail - Banking").
      - If no hyphens exist, treats the entire text as the AU Name with a blank Segment.
      
   4. DATA CLEANSING & ZERO-PADDING:
      - Strips the hidden ".0" ghost appended by Databricks/Excel imports.
      - ZERO-PADDING: Forces purely numeric Cost Center IDs under 4 digits to be 
        exactly 4 digits (e.g., '419' -> '0419') without truncating 4+ digit IDs.
      - Strips rogue leading/trailing double quotes (caused by CSV escaping).
      - Converts double/multiple spaces into a single space.
      - Strictly drops any ghost rows where the AU_ID or AU_Name is missing/blank.
      
   5. SINGLE SOURCE OF TRUTH (Deduplication & Standardization):
      - Uses UNION and DISTINCT to remove duplicate AU mappings per Cost Center.
      - Resolves naming conflicts across different Cost Centers (e.g., "Insurance" vs 
        "General Insurance" for the same ID) by using a Window Function to force a 
        single, standardized AU Name and Segment per AU_ID alphabetically.
=================================================================================== */

CREATE OR REPLACE TEMP VIEW vw_cost_center_mapping_bootstrap AS

WITH Raw_Import AS (
    -- 0a. Pull raw columns and strip the ".0" Excel ghost
    SELECT 
        REGEXP_REPLACE(TRIM(CAST(`CostCenterId` AS STRING)), '\\.0$', '') AS Raw_CC_String,
        TRIM(CAST(`AssessableUnitID` AS STRING)) AS Primary_AU_ID,
        TRIM(`AssessableUnitName`) AS Primary_AU_Name,
        TRIM(`Segment`) AS Primary_Segment,
        TRIM(`AdditionalAssessableUnitIDandNameandSegment`) AS Additional_Mixed_String,
        TRIM(`AdditionalAUID`) AS Additional_AU_ID
    FROM hive_metastore.ra_adido_2025.fy25_cost_center_mapping
),

Base_Data AS (
    -- 0b. Apply safe Zero-Padding (Only pads numbers under 4 digits)
    SELECT 
        CASE 
            WHEN Raw_CC_String RLIKE '^[0-9]+$' AND LENGTH(Raw_CC_String) < 4 
            THEN LPAD(Raw_CC_String, 4, '0')
            ELSE Raw_CC_String
        END AS Cost_Center_ID,
        Primary_AU_ID,
        Primary_AU_Name,
        Primary_Segment,
        Additional_Mixed_String,
        Additional_AU_ID
    FROM Raw_Import
),

Branch_Primary AS (
    -- LOGIC 1a: If Additional string is 'Yes', strictly use Primary
    SELECT 
        Cost_Center_ID,
        Primary_AU_ID AS AU_ID,
        Primary_AU_Name AS AU_Name,
        Primary_Segment AS Segment_Name
    FROM Base_Data
    WHERE UPPER(Additional_Mixed_String) = 'YES' 
       OR Additional_Mixed_String IS NULL 
       OR Additional_Mixed_String = ''
),

Branch_Additional_Raw AS (
    -- LOGIC 1b: If Additional string is NOT 'Yes', strictly use Additional
    SELECT 
        Cost_Center_ID,
        CONCAT_WS(' ', COALESCE(Additional_Mixed_String, ''), COALESCE(Additional_AU_ID, '')) AS Mashed_String
    FROM Base_Data
    WHERE UPPER(Additional_Mixed_String) != 'YES' 
      AND Additional_Mixed_String IS NOT NULL 
      AND Additional_Mixed_String != ''
),

Extracted_Blocks AS (
    -- LOGIC 2: Slice mashed text at 6-digit boundaries
    SELECT 
        Cost_Center_ID,
        EXPLODE(regexp_extract_all(Mashed_String, '([0-9]{6}.*?(?=[0-9]{6}|$))')) AS Raw_Block
    FROM Branch_Additional_Raw
    WHERE Mashed_String != ''
),

Separated_ID_And_Remainder AS (
    -- LOGIC 3a: Extract 6-digit ID and remainder
    SELECT 
        Cost_Center_ID,
        TRIM(regexp_extract(Raw_Block, '^([0-9]{6})', 1)) AS AU_ID,
        TRIM(REGEXP_REPLACE(Raw_Block, '^[0-9]{6}[ \t-]*', '')) AS Remainder
    FROM Extracted_Blocks
    WHERE TRIM(Raw_Block) != ''
),

Parsed_Additionals AS (
    -- LOGIC 3b: Parse remainder based on hyphens
    SELECT 
        Cost_Center_ID,
        AU_ID,
        CASE 
            WHEN Remainder LIKE '%-%' THEN TRIM(regexp_extract(Remainder, '^(.*)[ \t]*-[ \t]*[^-]+$', 1))
            ELSE Remainder 
        END AS AU_Name,
        CASE 
            WHEN Remainder LIKE '%-%' THEN TRIM(regexp_extract(Remainder, '.*[ \t]*-[ \t]*([^-]+)$', 1))
            ELSE '' 
        END AS Segment_Name
    FROM Separated_ID_And_Remainder
),

Cleaned_Stack AS (
    -- LOGIC 4: Combine branches, clean formatting
    SELECT DISTINCT 
        Cost_Center_ID, 
        AU_ID, 
        TRIM(REGEXP_REPLACE(REGEXP_REPLACE(AU_Name, '^"|"$', ''), '[ ]+', ' ')) AS AU_Name, 
        TRIM(REGEXP_REPLACE(REGEXP_REPLACE(Segment_Name, '^"|"$', ''), '[ ]+', ' ')) AS Segment_Name 
    FROM (
        SELECT Cost_Center_ID, AU_ID, AU_Name, Segment_Name FROM Branch_Primary
        UNION
        SELECT Cost_Center_ID, AU_ID, AU_Name, Segment_Name FROM Parsed_Additionals
    )
    WHERE AU_ID IS NOT NULL AND AU_ID != ''
      AND AU_Name IS NOT NULL AND TRIM(AU_Name) != ''
)

-- LOGIC 5: Final Standardization
SELECT DISTINCT 
    Cost_Center_ID,
    AU_ID,
    FIRST_VALUE(AU_Name) OVER (PARTITION BY AU_ID ORDER BY AU_Name ASC) AS AU_Name,
    FIRST_VALUE(Segment_Name) OVER (PARTITION BY AU_ID ORDER BY AU_Name ASC) AS Segment_Name
FROM Cleaned_Stack;

In [ ]:
%sql
/* ===================================================================================
   DEBUG SIMULATOR: Cost Center Mapping Pipeline Trace
   PURPOSE: Exposes the internal routing, zero-padding, and Regex extraction steps 
   to troubleshoot missing or incorrectly mapped Cost Centers.
=================================================================================== */

WITH Raw_Import AS (
    SELECT 
        REGEXP_REPLACE(TRIM(CAST(`CostCenterId` AS STRING)), '\\.0$', '') AS Raw_CC_String,
        TRIM(CAST(`AssessableUnitID` AS STRING)) AS Primary_AU_ID,
        TRIM(`AssessableUnitName`) AS Primary_AU_Name,
        TRIM(`AdditionalAssessableUnitIDandNameandSegment`) AS Additional_Mixed_String,
        TRIM(`AdditionalAUID`) AS Additional_AU_ID
    FROM hive_metastore.ra_adido_2025.fy25_cost_center_mapping
),

Base_Data AS (
    SELECT 
        Raw_CC_String,
        CASE 
            WHEN Raw_CC_String RLIKE '^[0-9]+$' AND LENGTH(Raw_CC_String) < 4 
            THEN LPAD(Raw_CC_String, 4, '0')
            ELSE Raw_CC_String
        END AS Padded_CC_ID,
        Primary_AU_ID,
        Primary_AU_Name,
        Additional_Mixed_String,
        Additional_AU_ID
    FROM Raw_Import
),

-- ============================================================================
-- BRANCH 1: PRIMARY ("YES") ROUTING SIMULATION
-- ============================================================================
Branch_Primary_Sim AS (
    SELECT 
        Raw_CC_String,
        Padded_CC_ID,
        Additional_Mixed_String AS Raw_Additional_String,
        'PRIMARY BRANCH (Cols B,C,D)' AS Routing_Decision,
        'N/A - Direct Pull' AS String_Fed_To_Regex,
        'N/A' AS Regex_Raw_Block,
        Primary_AU_ID AS Final_Mapped_AU,
        Primary_AU_Name AS Final_Mapped_Name,
        'SUCCESS' AS Status
    FROM Base_Data
    WHERE UPPER(Additional_Mixed_String) = 'YES' 
       OR Additional_Mixed_String IS NULL 
       OR Additional_Mixed_String = ''
),

-- ============================================================================
-- BRANCH 2: ADDITIONAL (REGEX) ROUTING SIMULATION
-- ============================================================================
Branch_Additional_Prep AS (
    SELECT 
        Raw_CC_String,
        Padded_CC_ID,
        Additional_Mixed_String AS Raw_Additional_String,
        'ADDITIONAL BRANCH (Regex Slice)' AS Routing_Decision,
        CONCAT_WS(' ', COALESCE(Additional_Mixed_String, ''), COALESCE(Additional_AU_ID, '')) AS String_Fed_To_Regex
    FROM Base_Data
    WHERE UPPER(Additional_Mixed_String) != 'YES' 
      AND Additional_Mixed_String IS NOT NULL 
      AND Additional_Mixed_String != ''
),

-- We use LEFT JOIN with EXPLODE so that if Regex fails to find a 6-digit ID, 
-- the row doesn't vanish. It stays visible so we can flag it as an error!
Branch_Additional_Explode AS (
    SELECT 
        p.*,
        blocks.Raw_Block AS Regex_Raw_Block
    FROM Branch_Additional_Prep p
    LEFT JOIN LATERAL (
        SELECT EXPLODE(regexp_extract_all(p.String_Fed_To_Regex, '([0-9]{6}.*?(?=[0-9]{6}|$))')) AS Raw_Block
    ) blocks ON true
),

Branch_Additional_Sim AS (
    SELECT 
        Raw_CC_String,
        Padded_CC_ID,
        Raw_Additional_String,
        Routing_Decision,
        String_Fed_To_Regex,
        COALESCE(Regex_Raw_Block, 'NULL') AS Regex_Raw_Block,
        TRIM(regexp_extract(Regex_Raw_Block, '^([0-9]{6})', 1)) AS Final_Mapped_AU,
        TRIM(REGEXP_REPLACE(Regex_Raw_Block, '^[0-9]{6}[ \t-]*', '')) AS Final_Mapped_Name,
        
        -- Status Flagging
        CASE 
            WHEN Regex_Raw_Block IS NULL THEN '❌ DROPPED: Regex found NO 6-digit IDs'
            WHEN TRIM(regexp_extract(Regex_Raw_Block, '^([0-9]{6})', 1)) = '' THEN '❌ DROPPED: Invalid ID Extraction'
            ELSE 'SUCCESS' 
        END AS Status
    FROM Branch_Additional_Explode
),

-- ============================================================================
-- COMBINE AND OUTPUT TRACE
-- ============================================================================
Combined_Sim AS (
    SELECT * FROM Branch_Primary_Sim
    UNION ALL
    SELECT * FROM Branch_Additional_Sim
)

SELECT * FROM Combined_Sim

-- 💡 TROUBLESHOOTING: Uncomment the line below to test a specific Cost Center!
-- WHERE Padded_CC_ID = '2830' 

ORDER BY Padded_CC_ID, Routing_Decision;